# itero: every Wikipedia edit, live

Wikimedia streams every edit to every Wikipedia as it happens, some five a second. itero turns the stream into events, counts what each 10 seconds bring, spots pages people edit again and again, and loses nothing when you pull the plug. Run the cells in order.

**1. Install itero** (and `jq`, which the example uses).

In [ ]:
!command -v jq >/dev/null || (apt-get -qq update && apt-get -qq install -y jq) >/dev/null
!command -v itero >/dev/null || curl -fsSL https://itero.sh/install | sh -s -- --to /usr/local/bin
!itero --version

**2. Get the example.** A notebook has no desktop, so notifications go to a file.

In [ ]:
!test -d /tmp/itero || git clone -q --depth 1 https://github.com/aegea-dev/itero /tmp/itero
%cd /tmp/itero/examples/wiki-stream
%env ITERO_NOTIFY_LOG=/tmp/notifications.log
%env ITERO_NO_SECRET_STORE=1

**3. The two workflows.** `pulse` keeps each 10 seconds of edits for one run (`batch`); `busy-page` tells a page people edited 3 times within 5 minutes (`at_least`, `per` page). Both drop an edit seen twice (`unique`).

In [ ]:
!cat workflows/pulse.json workflows/busy-page.json

**4. Start it, and watch 30 seconds of the world editing.**

In [ ]:
%%bash --bg
# runs on in the background; its pid, to stop it
echo $$ >/tmp/itero.pid
exec itero ws daemon --no-serve >/tmp/daemon.log 2>&1

In [ ]:
!sleep 30
!cat /tmp/notifications.log

**5. Pull the plug.** The connector noted the time of the last edit it told; edits go on without us.

In [ ]:
!kill $(cat /tmp/itero.pid)
!echo "last edit told: $(cat .itero/wiki.edited.since)"
!sleep 20

**6. Start it again.** It asks the stream for what came since: the first summary has what came while it was down, and an edit sent twice is dropped.

In [ ]:
%%bash --bg
echo $$ >/tmp/itero.pid
exec itero ws daemon --no-serve >/tmp/daemon2.log 2>&1

In [ ]:
!sleep 15
!grep resuming /tmp/daemon2.log
!tail -3 /tmp/notifications.log

**7. Stop it.**

In [ ]:
!kill $(cat /tmp/itero.pid)